# HOD-GNN: a GNN that learns from its own derivatives

**The idea in one sentence.** The derivative of a GNN's output with respect to its own input is a structure-aware quantity — for a base network computing $A^3X$, the diagonal $\partial h_v / \partial x_v$ counts the **triangles** at $v$ — so HOD-GNN propagates the exact Jacobian of a small base GIN *analytically alongside its features* and feeds those derivative channels, as learned structural encodings, to a downstream GNN.

This is the empirical architecture of **"On the Expressive Power of GNN Derivatives"** (Eitan, Eliasof, Gelberg, Frasca, Bar-Shalom & Maron, ICLR 2026, [arXiv:2510.02565](https://arxiv.org/abs/2510.02565)); to our knowledge this is its first public implementation. Message passing alone is bounded by 1-WL and provably cannot count triangles — the derivative features restore exactly what 1-WL is blind to. Theorem 4.2 of the paper makes the bridge to hand-crafted encodings precise, and this notebook lets you touch it:

1. at initialisation the derivative diagonals **equal the random-walk structural encoding (RWSE)** — then training makes them *learnable*;
2. the features **separate graphs 1-WL cannot** ($C_6$ vs $2{\times}C_3$);
3. everything is **batch-aware** (disjoint graphs never mix);
4. the backbone trains in the TopoBench pipeline like any other model.

## 1. The theorem you can touch: derivative diagonals = RWSE at init

Under the paper's Appendix-F initialisation (`structural_init`: constant-one projected input, identity base MLPs) with the row-normalised **mean** aggregation — the operator used in the constructive proof of Theorem 4.2 — the layer-$t$ diagonal Jacobian of node $v$ is exactly $(P^t)_{vv}$, the probability that a $t$-step random walk returns home. That is precisely the RWSE of GraphGPS, obtained here not as a precomputed feature but as the *starting point* of a fully learnable module. (The internal features carry a $1/t!$ residual scaling — Eq. 73 of the paper — which we undo below for the comparison.)

In [1]:
import math

import torch
from torch_geometric.data import Batch, Data

from topobench.nn.backbones.graph.hod_gnn import HODGNN

torch.manual_seed(0)


def make_graph(num_nodes, chords, feat_dim=8):
    ring = [(i, (i + 1) % num_nodes) for i in range(num_nodes)]
    src, dst = zip(*(ring + chords))
    ei = torch.tensor([src + dst, dst + src])  # undirected
    return Data(x=torch.randn(num_nodes, feat_dim), edge_index=ei)


def diag_blocks(model, data):
    """Per-layer [n, k, k] diagonal Jacobian blocks, t! rescaled."""
    n, k = data.num_nodes, model.base_channels
    idx_local = torch.arange(n)  # single graph: local index = global
    adj = model._adjacency(data.edge_index, n, None, torch.double)
    with torch.no_grad():
        _, diags = model._propagate_base(
            model.lin_in(data.x.double()), adj, idx_local, n
        )
    return [
        diags[:, t * k * k : (t + 1) * k * k].reshape(n, k, k)
        * math.factorial(t + 1)
        for t in range(model.base_layers)
    ]


model = HODGNN(
    in_channels=8, hidden_channels=32, out_channels=8,
    base_channels=3, base_layers=4,
    aggregation='mean', structural_init=True,
).double().eval()

g = make_graph(10, chords=[(0, 5), (2, 7), (3, 8)])
a = torch.zeros(10, 10, dtype=torch.double)
a[g.edge_index[1], g.edge_index[0]] = 1.0
p = a / a.sum(dim=1, keepdim=True)  # random-walk operator D^-1 A

for t, block in enumerate(diag_blocks(model, g), start=1):
    rwse_t = torch.diagonal(torch.linalg.matrix_power(p, t))
    ours_t = torch.diagonal(block, dim1=1, dim2=2)[:, 0]
    print(f'{t}-step: diagonal == RW return probability:',
          torch.allclose(ours_t, rwse_t, atol=1e-12),
          '| e.g. node 0:', round(ours_t[0].item(), 4))

1-step: diagonal == RW return probability: True | e.g. node 0: 0.0
2-step: diagonal == RW return probability: True | e.g. node 0: 0.4444
3-step: diagonal == RW return probability: True | e.g. node 0: 0.0
4-step: diagonal == RW return probability: True | e.g. node 0: 0.2901


## 2. Beyond 1-WL: separating $C_6$ from $2{\times}C_3$

Both graphs are 2-regular with constant features, so 1-WL message passing sees identical neighbourhood multisets at every round and *no MPNN can tell them apart*. The 3-step derivative channel is the paper's motivating $A^3$ example: with **sum** (GIN) aggregation it counts closed 3-walks, which exist only around triangles — zero everywhere on $C_6$, exactly 2 on every node of a triangle.

In [2]:
def cycles(sizes, feat_dim=8):
    edges, offset = [], 0
    for s in sizes:
        edges += [(offset + i, offset + (i + 1) % s) for i in range(s)]
        offset += s
    src, dst = zip(*edges)
    ei = torch.tensor([src + dst, dst + src])
    return Data(x=torch.ones(offset, feat_dim), edge_index=ei)


wl_blind = HODGNN(
    in_channels=8, hidden_channels=32, out_channels=8,
    base_channels=3, base_layers=4,
    aggregation='sum', structural_init=True,
).double().eval()

block_c6 = diag_blocks(wl_blind, cycles([6]))[2]      # 3-step channel
block_2c3 = diag_blocks(wl_blind, cycles([3, 3]))[2]
print('C6      3-step diagonal (all nodes):',
      torch.diagonal(block_c6, dim1=1, dim2=2)[:, 0].tolist())
print('2 x C3  3-step diagonal (all nodes):',
      torch.diagonal(block_2c3, dim1=1, dim2=2)[:, 0].tolist())

C6      3-step diagonal (all nodes): [0.0, 0.0, 0.0, 0.0, 0.0, 0.0]
2 x C3  3-step diagonal (all nodes): [2.0, 2.0, 2.0, 2.0, 2.0, 2.0]


## 3. Batch-aware by construction

The Jacobian's source axis is indexed *locally per graph*, so batching disjoint graphs (TopoBench batches 16 at a time) never leaks structure across graphs — a correctness property the challenge's triangle-counting task is very sensitive to.

In [3]:
g_a, g_b = make_graph(10, [(0, 5)]), make_graph(7, [(1, 4)])
batch = Batch.from_data_list([g_a, g_b])
with torch.no_grad():
    out_batch = model(batch.x.double(), batch.edge_index, batch.batch)
    out_alone = model(g_a.x.double(), g_a.edge_index)
diff = (out_batch[:10] - out_alone).abs().max().item()
print('graph A alone == graph A in a batch:', diff < 1e-12,
      f'(max abs diff {diff:.2e})')

graph A alone == graph A in a batch: True (max abs diff 2.78e-17)


## 4. Training in the TopoBench pipeline

The model ships with `configs/model/graph/hod_gnn.yaml`, so the canonical run is `python -m topobench model=graph/hod_gnn dataset=<dataset>`. The config defaults to the **mean** (Theorem-4.2) aggregation with `structural_init` — the combination whose derivative features stay in $[0,1]$ and train stably; with sum aggregation the same init produces walk-count-scale features ($(A^t)_{vv}$) that destabilise training on dense graphs under a hot learning rate. Here we train briefly on MUTAG on CPU.

In [4]:
import os
from pathlib import Path

from hydra import compose, initialize_config_dir
from hydra.core.global_hydra import GlobalHydra

from topobench.run import run
from topobench.utils.config_resolvers import register_all_resolvers

root = Path.cwd().resolve()
while not (root / 'configs' / 'run.yaml').exists():
    root = root.parent
os.environ['PROJECT_ROOT'] = str(root)
out_dir = root / 'logs' / 'tutorial_hod_gnn'
out_dir.mkdir(parents=True, exist_ok=True)

register_all_resolvers()
GlobalHydra.instance().clear()
with initialize_config_dir(version_base='1.3', config_dir=str(root / 'configs')):
    cfg = compose(
        config_name='run.yaml',
        overrides=[
            'model=graph/hod_gnn',
            'dataset=graph/MUTAG',
            'logger=csv',
            'trainer.max_epochs=10',
            'trainer.accelerator=cpu',
            'trainer.devices=1',
            '+trainer.enable_progress_bar=False',
            f'paths.output_dir={out_dir.as_posix()}',
            f'paths.work_dir={root.as_posix()}',
        ],
    )

metric_dict, _ = run(cfg)
print('\nMetrics:')
for key, value in metric_dict.items():
    print(f'{key:<20s} {float(value):.4f}')

Seed set to 42


/home/aarav/topobench/.venv/lib/python3.11/site-packages/outdated/__init__.py:36: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import parse_version


Trainer already configured with model summary callbacks: [<class 'lightning.pytorch.callbacks.rich_model_summary.RichModelSummary'>]. Skipping setting a default `ModelSummary` callback.


GPU available: True (cuda), used: False


TPU available: False, using: 0 TPU cores


...

Validation: |          | 0/? [00:00<?, ?it/s]

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃      Validate metric      ┃       DataLoader 0        ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│       val/accuracy        │    0.6808510422706604     │
│         val/auroc         │    0.7937500476837158     │
│          val/f1           │    0.40506330132484436    │
│         val/loss          │    0.5873425602912903     │
│       val/precision       │    0.3404255211353302     │
│        val/recall         │            0.5            │
└───────────────────────────┴───────────────────────────┘

Testing: |          | 0/? [00:00<?, ?it/s]

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃        Test metric        ┃       DataLoader 0        ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│       test/accuracy       │    0.6382978558540344     │
│        test/auroc         │    0.9666666984558105     │
│          test/f1          │    0.3896103799343109     │
│         test/loss         │    0.5789005160331726     │
│      test/precision       │    0.3191489279270172     │
│        test/recall        │            0.5            │
└───────────────────────────┴───────────────────────────┘


Metrics:
lr-Adam              0.0010
train/accuracy       0.6702
train/auroc          0.8725
train/f1             0.4013
train/precision      0.3351
train/recall         0.5000
val/loss             0.5623
best_epoch           1.0000
best_epoch/val/loss  0.5825
val/accuracy         0.6809
val/auroc            0.8125
val/f1               0.4051
val/precision        0.3404
val/recall           0.5000
train/loss           0.5672
best_epoch/train/accuracy 0.6702
best_epoch/train/auroc 0.7066
best_epoch/train/f1  0.4013
best_epoch/train/precision 0.3351
best_epoch/train/recall 0.5000
best_epoch/train/loss 0.6083
best_epoch/val/accuracy 0.6809
best_epoch/val/auroc 0.7938
best_epoch/val/f1    0.4051
best_epoch/val/precision 0.3404
best_epoch/val/recall 0.5000


We let a GNN differentiate itself: the exact Jacobian of a small base GIN, propagated analytically and end-to-end differentiably, supplies each node with learnable structural features that *start* at the classical random-walk encoding and go strictly beyond 1-WL. On the challenge's GraphUniverse grid this cuts the triangle-counting error by roughly an order of magnitude versus a strong graph-transformer baseline while improving out-of-distribution community detection — see `2026_tdl_challenge/outputs/` and the accompanying PR for the full 72-run results. To benchmark it yourself, set `MODEL_CONFIG = "graph/hod_gnn"` in the challenge evaluation pipeline.